# 07 — Drift check on hand written sentences (all intent models)

Every intent model scores 0.99+ on the Bitext test split because the corpus is
template-generated. This notebook asks the drift question instead: what happens on
wording the templates never produced? It runs the four stress tests from
`v3/docs/final/experiments/generalization.py`, using that script's sets, perturbations,
seed and scoring. It re-measures the report's six models live and adds XLNet, Laya and SetFit
from notebook 06.

| Test | What | Size |
|---|---|---:|
| hand written | sentences the team wrote after training, 4 per intent (`content/generalization_sets.yaml`) | 108 |
| padded | test split with 1–2 polite courtesy sentences around each message | 3,604 |
| noise | test split with character typos at 3–20% of letters | 3,604 × 5 |
| unknown | internet provider messages no Bitext intent covers | 30 |

The report's own `data/generalization.json` is not written here.

In [1]:
import sys, json, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import common  # noqa: F401  (interpreter guard, CUDA shim)
import pandas as pd
import torch

import latest_generalization as lg
g = lg.g
print(f"hand written sentences: {sum(len(v) for v in g.sets['paraphrases'].values())}, "
      f"unknown: {len(g.sets['isp'])}, test split: {len(g.test)}")

hand written sentences: 108, unknown: 30, test split: 3604


## 1. The report's six models (saved notebook-01 artifacts, never retrained)

In [2]:
holder = g.Models()
base = lg.evaluate(holder, g.NAMES)
del holder; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

noise 0.0 {'tfidf_svc': 0.999, 'tfidf_logreg': 0.991, 'minilm_logreg': 0.992, 'minilm_ft': 0.997, 'distilbert_ft': 0.996, 'minilm_zeroshot': 0.728}


noise 0.03 {'tfidf_svc': 0.995, 'tfidf_logreg': 0.952, 'minilm_logreg': 0.949, 'minilm_ft': 0.969, 'distilbert_ft': 0.966, 'minilm_zeroshot': 0.641}


noise 0.06 {'tfidf_svc': 0.981, 'tfidf_logreg': 0.892, 'minilm_logreg': 0.877, 'minilm_ft': 0.926, 'distilbert_ft': 0.923, 'minilm_zeroshot': 0.558}


noise 0.1 {'tfidf_svc': 0.96, 'tfidf_logreg': 0.799, 'minilm_logreg': 0.762, 'minilm_ft': 0.843, 'distilbert_ft': 0.831, 'minilm_zeroshot': 0.465}


noise 0.15 {'tfidf_svc': 0.923, 'tfidf_logreg': 0.674, 'minilm_logreg': 0.635, 'minilm_ft': 0.739, 'distilbert_ft': 0.716, 'minilm_zeroshot': 0.362}


noise 0.2 {'tfidf_svc': 0.864, 'tfidf_logreg': 0.551, 'minilm_logreg': 0.515, 'minilm_ft': 0.63, 'distilbert_ft': 0.611, 'minilm_zeroshot': 0.283}


## 2. XLNet fine-tuned, Laya and SetFit (notebook 06)

In [3]:
holder = lg.Latest()
latest = lg.evaluate(holder, lg.NAMES)
del holder; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

D:\DSEP22\model testing\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape
D:\DSEP22\model testing\.venv\Lib\site-packages\joblib\numpy_pickle.py:204: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape[::-1]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

D:\DSEP22\model testing\.venv\Lib\site-packages\joblib\numpy_pickle.py:207: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape
D:\DSEP22\model testing\.venv\Lib\site-packages\joblib\numpy_pickle.py:204: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  array.shape = self.shape[::-1]


...

noise 0.03 {'xlnet_ft': 0.974, 'laya_zeroshot': 0.843, 'laya_encoder_logreg': 0.877, 'setfit_k8': 0.833, 'setfit_k64': 0.891}


noise 0.06 {'xlnet_ft': 0.945, 'laya_zeroshot': 0.775, 'laya_encoder_logreg': 0.751, 'setfit_k8': 0.725, 'setfit_k64': 0.801}


noise 0.1 {'xlnet_ft': 0.87, 'laya_zeroshot': 0.667, 'laya_encoder_logreg': 0.582, 'setfit_k8': 0.609, 'setfit_k64': 0.679}


noise 0.15 {'xlnet_ft': 0.768, 'laya_zeroshot': 0.559, 'laya_encoder_logreg': 0.426, 'setfit_k8': 0.486, 'setfit_k64': 0.557}


noise 0.2 {'xlnet_ft': 0.653, 'laya_zeroshot': 0.454, 'laya_encoder_logreg': 0.306, 'setfit_k8': 0.395, 'setfit_k64': 0.446}


## 3. Results

In [4]:
rows = []
for n, d in lg.table([base, latest]):
    p, u = d["paraphrase"][n], d["unknown"][n]
    rows.append({
        "model": n,
        "test_f1": d["in_distribution"][n]["macro_f1"],
        "hand_written_f1": p["macro_f1"],
        "hand_written_lenient_acc": p["lenient_accuracy"],
        "drop": d["in_distribution"][n]["macro_f1"] - p["macro_f1"],
        "padded_f1": d["padded"][n]["macro_f1"],
        "noise10_f1": d["noise"]["0.1"][n]["macro_f1"],
        "noise20_f1": d["noise"]["0.2"][n]["macro_f1"],
        "unknown_auroc": u["auroc"],
        "unknown_accepted": u["accepted"],
    })
summary = pd.DataFrame(rows).round(3)
print(summary.to_string(index=False))

              model  test_f1  hand_written_f1  hand_written_lenient_acc  drop  padded_f1  noise10_f1  noise20_f1  unknown_auroc  unknown_accepted
      laya_zeroshot    0.907            0.715                     0.806 0.192      0.886       0.667       0.454          0.900             0.400
      minilm_logreg    0.992            0.680                     0.806 0.312      0.963       0.762       0.515          0.996             0.000
         setfit_k64    0.969            0.657                     0.769 0.311      0.953       0.679       0.446          0.995             0.000
          setfit_k8    0.927            0.609                     0.731 0.318      0.901       0.609       0.395          0.991             0.000
          minilm_ft    0.997            0.574                     0.704 0.423      0.980       0.843       0.630          0.997             0.000
           xlnet_ft    0.997            0.552                     0.685 0.445      0.813       0.870       0.653          0.

### Typo sweep (macro-F1 by character error rate)

In [5]:
noise = pd.DataFrame({rate: {**{n: v["macro_f1"] for n, v in base["noise"][rate].items()},
                             **{n: v["macro_f1"] for n, v in latest["noise"][rate].items()}}
                      for rate in base["noise"]}).round(3)
print(noise.to_string())

                       0.0   0.03   0.06    0.1   0.15    0.2
tfidf_svc            0.999  0.995  0.981  0.960  0.923  0.864
tfidf_logreg         0.991  0.952  0.892  0.799  0.674  0.551
minilm_logreg        0.992  0.949  0.877  0.762  0.635  0.515
minilm_ft            0.997  0.969  0.926  0.843  0.739  0.630
distilbert_ft        0.996  0.966  0.923  0.831  0.716  0.611
minilm_zeroshot      0.728  0.641  0.558  0.465  0.362  0.283
xlnet_ft             0.997  0.974  0.945  0.870  0.768  0.653
laya_zeroshot        0.907  0.843  0.775  0.667  0.559  0.454
laya_encoder_logreg  0.974  0.877  0.751  0.582  0.426  0.306
setfit_k8            0.927  0.833  0.725  0.609  0.486  0.395
setfit_k64           0.969  0.891  0.801  0.679  0.557  0.446


### Selective answering on the hand written set
Answer only when confidence clears the 5th percentile of in-distribution confidence.
`confidently_wrong` is the share of all 108 sentences answered confidently and wrongly.

In [6]:
sel = pd.DataFrame({**base["paraphrase_selective"], **latest["paraphrase_selective"]}).T.round(3)
print(sel.to_string())

                     coverage  accuracy_when_answering  confidently_wrong
tfidf_svc               0.037                    1.000              0.000
tfidf_logreg            0.102                    0.818              0.019
minilm_logreg           0.269                    0.897              0.028
minilm_ft               0.222                    0.750              0.056
distilbert_ft           0.074                    0.750              0.019
minilm_zeroshot         0.917                    0.545              0.417
xlnet_ft                0.111                    0.917              0.009
laya_zeroshot           0.954                    0.738              0.250
laya_encoder_logreg     0.509                    0.764              0.120
setfit_k8               0.528                    0.754              0.130
setfit_k64              0.343                    0.838              0.056


### Where each model goes wrong (hand written set)

In [7]:
models = base["models"] + latest["models"]
pred = pd.DataFrame([{"text": b["text"], "true": b["true"],
                      **{n: b[n] for n in base["models"]},
                      **{n: l[n] for n in latest["models"]}}
                     for b, l in zip(base["paraphrase_rows"], latest["paraphrase_rows"])])
per_intent = pd.DataFrame({n: (pred[n] == pred["true"]).groupby(pred["true"]).mean() for n in models}).round(2)
print(per_intent.to_string())

                          tfidf_svc  tfidf_logreg  minilm_logreg  minilm_ft  distilbert_ft  minilm_zeroshot  xlnet_ft  laya_zeroshot  laya_encoder_logreg  setfit_k8  setfit_k64
true                                                                                                                                                                            
cancel_order                   0.25          0.50           0.75       0.50           0.25             0.50      0.75           0.75                 0.75       0.75        0.25
change_order                   0.25          0.50           1.00       1.00           1.00             0.75      0.75           1.00                 0.25       0.75        1.00
change_shipping_address        0.00          0.00           0.75       0.50           0.50             0.75      0.25           0.75                 0.00       0.00        0.25
check_cancellation_fee         0.25          0.25           0.50       0.25           0.25             0.25      0.

In [8]:
wrong_everywhere = pred[[all(pred.loc[i, n] != pred.loc[i, "true"] for n in models) for i in pred.index]]
print(f"{len(wrong_everywhere)} sentences no model gets right:")
print(wrong_everywhere[["true", "text"]].to_string(index=False))

3 sentences no model gets right:
                  true                                                                                                                                    text
check_cancellation_fee                                                                                       If I drop my order now, will it cost me anything?
             complaint                                                                                 Something went badly wrong and I want somebody to know.
           track_order It has been five days since I paid and I have heard nothing, so could you tell me where my package is and when the courier will arrive.


In [9]:
out = Path.cwd().parent / "results"
summary.to_csv(out / "summary_generalization_all.csv", index=False)
(out / "latest_generalization.json").write_text(json.dumps(latest, indent=1), encoding="utf-8")
lg.write_report([base, latest])

# Hand written generalization: XLNet and Laya vs every intent model

Sets, perturbations and scoring from `v3/docs/final/experiments/generalization.py`.

| Model | In-dist F1 | Hand written F1 | Hand written lenient acc | Drop | Padded F1 | Noise 10% F1 | Noise 20% F1 | Unknown AUROC | Unknown accepted |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| laya_zeroshot | 0.907 | **0.715** | 0.806 | 0.192 | 0.886 | 0.667 | 0.454 | 0.900 | 40% |
| minilm_logreg | 0.992 | **0.680** | 0.806 | 0.312 | 0.963 | 0.762 | 0.515 | 0.996 | 0% |
| setfit_k64 | 0.969 | **0.657** | 0.769 | 0.311 | 0.953 | 0.679 | 0.446 | 0.995 | 0% |
| setfit_k8 | 0.927 | **0.609** | 0.731 | 0.318 | 0.901 | 0.609 | 0.395 | 0.991 | 0% |
| minilm_ft | 0.997 | **0.574** | 0.704 | 0.423 | 0.980 | 0.843 | 0.630 | 0.997 | 0% |
| xlnet_ft | 0.997 | **0.552** | 0.685 | 0.445 | 0.813 | 0.870 | 0.653 | 0.998 | 0% |
| minilm_zeroshot | 0.728 | **0.530** | 0.639 | 0.198 | 0.658 | 0.465 | 0.283 | 0.913 | 47% |
| laya_encoder_log